# VideoMAE alternative-encoder experiment

Recreated from `VideoMAE.ipynb`. Shared model definitions are imported from `src/sign_language_lab/`; training and evaluation remain visible below. Outputs were cleared during organization.

**Status:** historical experiment with unresolved model and evaluation issues. Formatting and module extraction do not validate its results. Read [the experiment review](../docs/experiment_review.md) and [setup guide](../docs/reproduction.md) before executing cells.

Cell metadata records original cell numbers. Dataset and checkpoint paths now use environment-based configuration.

## Repository and path setup

In [ ]:
import sys
from pathlib import Path

repository_root = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "src" / "sign_language_lab").is_dir()
    ),
    None,
)
if repository_root is None:
    raise RuntimeError("Open this notebook from inside the repository.")
source_directory = str(repository_root / "src")
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

from sign_language_lab.config import ExperimentPaths

paths = ExperimentPaths.from_environment("videomae_experiment")
paths.output_dir.mkdir(parents=True, exist_ok=True)


## Imports and dataset paths

In [ ]:
import av
import numpy as np
import torch
import av
import cv2
import torch
import numpy as np
import mediapipe as mp
from typing import List

excel_file = str(paths.annotations_file)
train_dir = str(paths.train_dir)
test_dir = str(paths.test_dir)
val_dir = str(paths.dev_dir)


## Inspect the pretrained video backbone

In [ ]:
import torch
from transformers import AutoImageProcessor, VideoMAEModel
from torchinfo import summary
import numpy as np

# Create a proper dummy video input that mimics real video data
dummy_video = np.random.randint(
    0, 255, size=(16, 224, 224, 3), dtype=np.uint8
)  # (num_frames, height, width, channels)

image_processor = AutoImageProcessor.from_pretrained("MCG-NJU/videomae-base")
model = VideoMAEModel.from_pretrained("MCG-NJU/videomae-base")

# Process the input correctly through the image processor
inputs = image_processor(list(dummy_video), return_tensors="pt", do_rescale=True)

# Extract the pixel_values tensor from the processed inputs
pixel_values = inputs["pixel_values"]

# Now get the model summary using the tensor directly
model_summary = summary(
    model,
    input_data={
        "pixel_values": pixel_values
    },  # Pass as a dictionary with the expected key
    col_names=["input_size", "output_size", "num_params"],
)
print(model_summary)


## Attention components

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import VivitForVideoClassification
from sign_language_lab.attention import (
    HandUpperBodyAttention,
)  # [batch_size, num_frames, feature_dim]

from sign_language_lab.attention import SpatialAttention
from sign_language_lab.videomae_experiment import TemporalSkipConnection
from sign_language_lab.attention import PositionalEncoding
from sign_language_lab.attention import EnhancedPositionalEncoding


## Encoder implementation

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import VideoMAEModel

from sign_language_lab.videomae_experiment import VideoMAEEncoderWithTemporalModel


## Encoder summary

In [ ]:
import torch
from transformers import AutoImageProcessor, VideoMAEModel
from torchinfo import summary
import numpy as np

# Create a proper dummy video input
dummy_video = np.random.randint(0, 255, size=(16, 224, 224, 3), dtype=np.uint8)

# Initialize the image processor and model
image_processor = AutoImageProcessor.from_pretrained("MCG-NJU/videomae-base")
model = VideoMAEEncoderWithTemporalModel(
    pretrained_model_name="MCG-NJU/videomae-base",
    feature_dim=768,
    hidden_dim=512,
    temporal_model="bilstm",
    use_sliding_windows=True,
)

# Set up device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Process input and move to the same device
inputs = image_processor(list(dummy_video), return_tensors="pt", do_rescale=True)
pixel_values = inputs["pixel_values"].to(device)  # Move input tensor to the same device

# Get model summary
model_summary = summary(
    model,
    input_data={"pixel_values": pixel_values},
    col_names=["input_size", "output_size", "num_params"],
    device=device,  # Specify the device for torchinfo
)
print(model_summary)


## Batch-processing prototype

### `process_batch_for_videomae`

In [ ]:
def process_batch_for_videomae(video_files, image_processor, batch_size=5, clip_len=16):
    """
    Process a batch of videos for VideoMAE, including pose landmarks and proper preprocessing.
    """
    # Process initial batch of videos to get frames with pose landmarks
    video_batch = process_video_batch(
        video_files, clip_len=clip_len, frame_sample_rate=1
    )

    # Convert frames to the format expected by VideoMAE's image processor
    frame_list = []
    for video in video_batch:
        # Convert from [T, C, H, W] to [T, H, W, C]
        video_np = video.numpy()  # Convert to numpy
        video_np = np.transpose(video_np, (0, 2, 3, 1))  # Reorder dimensions
        # Ensure values are in uint8 range [0, 255]
        video_np = (video_np * 255).astype(np.uint8)
        frame_list.append(video_np)

    try:
        # Process with VideoMAE's image processor
        inputs = image_processor(
            frame_list,
            return_tensors="pt",
            do_rescale=True,  # Let image processor handle normalization
        )

        print(f"Processed inputs pixel values shape: {inputs.pixel_values.shape}")
        return inputs

    except Exception as e:
        print(f"Error in image processing: {str(e)}")
        raise


## Training annotations

In [ ]:
# RETRIVING VIDEO_FILES, AND SENTENCES
import pandas as pd
import glob
import os

video_files1 = glob.glob(
    os.path.join(train_dir, "*.mp4")
)  # Adjust extension as necessary

if video_files1:
    print(f"Found {len(video_files1)} video files.")
else:
    print("No video files found in the directory.")
    video_files1 = []  # Exit early if no video files found

# Step 2: Extract codes from video filenames
video_file_codes1 = [
    os.path.splitext(os.path.basename(video))[0].split("-")[-1].strip()
    for video in video_files1
]

# Step 3: Load Excel file and extract codes from the 'train' sheet
try:
    df_combined = pd.read_excel(
        excel_file, sheet_name="train"
    )  # Adjust sheet name as necessary
    df_combined["code"] = df_combined["name"].apply(lambda x: x.split("-")[-1].strip())

    # Step 4: Match video file codes with codes from the Excel file
    matching_data = df_combined[df_combined["code"].isin(video_file_codes1)]

    if not matching_data.empty:
        matched_translations1 = []
        matched_video_files1 = []

        # Iterate over video file codes to match properly
        for video_code in video_file_codes1:
            matching_row = matching_data[matching_data["code"] == video_code]
            if not matching_row.empty:
                matched_translations1.append(matching_row["gloss"].iloc[0])
                matched_video_files1.append(video_code)

    else:
        # Handle cases with no matches
        unmatched_video_codes = [
            code
            for code in video_file_codes2
            if code not in df_combined["code"].tolist()
        ]

except FileNotFoundError:
    print(f"Error: Excel file '{excel_file}' not found.")
except KeyError as e:
    print(f"Error: Missing expected column in Excel file. Details: {e}")
# print((matched_translations1[:100]))
# print(len(video_files1[:5]))


## Video preprocessing prototype

In [ ]:
import av
import torch
import numpy as np

# from transformers import AutoImageProcessor, VideoMAEEncoderWithTemporalModel
from typing import List, Tuple


### `read_video_pyav`

In [ ]:
def read_video_pyav(container, indices):
    """
    Decode the video with PyAV decoder.
    Args:
        container (`av.container.input.InputContainer`): PyAV container.
        indices (`List[int]`): List of frame indices to decode.
    Returns:
        result (np.ndarray): np array of decoded frames of shape (num_frames, height, width, 3).
    """
    frames = []
    container.seek(0)
    start_index = indices[0]
    end_index = indices[-1]

    for i, frame in enumerate(container.decode(video=0)):
        if i > end_index:
            break
        if i >= start_index and i in indices:
            frames.append(frame)

    # Stack frames directly into numpy array with rgb24 format
    frames_array = np.stack([x.to_ndarray(format="rgb24") for x in frames])
    print(f"Frames array shape: {frames_array.shape}")  # Debug print
    return frames_array


### `sample_frame_indices`

In [ ]:
def sample_frame_indices(clip_len, frame_sample_rate, seg_len):
    """
    Sample a given number of frame indices from the video.
    Args:
        clip_len (int): Total number of frames to sample.
        frame_sample_rate (int): Sample every n-th frame.
        seg_len (int): Maximum allowed index of sample's last frame.
    Returns:
        indices (np.ndarray): Array of sampled frame indices
    """
    converted_len = int(clip_len * frame_sample_rate)
    end_idx = np.random.randint(converted_len, seg_len)
    start_idx = end_idx - converted_len
    indices = np.linspace(start_idx, end_idx, num=clip_len)
    indices = np.clip(indices, start_idx, end_idx - 1).astype(np.int64)
    return indices


### `process_video_batch`

In [ ]:
def process_video_batch(video_path: str, image_processor, clip_len: int = 16):
    """
    Process a single video file for VideoMAE.
    """
    # Open the video file
    container = av.open(video_path)

    # Get total number of frames
    total_frames = container.streams.video[0].frames
    print(f"Total frames in video: {total_frames}")

    # Sample frame indices
    indices = sample_frame_indices(
        clip_len=clip_len, frame_sample_rate=1, seg_len=total_frames
    )

    # Read the video frames
    video = read_video_pyav(container, indices)

    # Process with image processor
    inputs = image_processor(list(video), return_tensors="pt")

    # Close the video container
    container.close()

    return inputs


### `test_videomae_on_videos`

In [ ]:
def test_videomae_on_videos(
    model: nn.Module,
    image_processor,
    video_paths: List[str],
    batch_size: int = 2,
    device: str = "cuda",
):
    """
    Process a batch of videos using VideoMAE.
    """
    model = model.to(device)
    model.eval()

    all_features = []

    for i in range(0, len(video_paths), batch_size):
        batch_paths = video_paths[i : i + batch_size]
        print(
            f"\nProcessing batch {i//batch_size + 1}/{(len(video_paths) + batch_size - 1)//batch_size}"
        )

        batch_inputs = []
        for video_path in batch_paths:
            try:
                inputs = process_video_batch(video_path, image_processor)
                batch_inputs.append(inputs["pixel_values"])
            except Exception as e:
                print(f"Error processing video {video_path}: {str(e)}")
                continue

        if batch_inputs:
            # Stack batch inputs
            batch_tensor = torch.cat(batch_inputs, dim=0).to(device)

            # Process through model
            with torch.no_grad():
                features = model(pixel_values=batch_tensor)

            all_features.append(features.cpu())

    if all_features:
        all_features = torch.cat(all_features, dim=0)
        return all_features
    else:
        raise ValueError("No features were successfully extracted")


# Example usage


In [ ]:
if __name__ == "__main__":
    # Initialize components
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    image_processor = AutoImageProcessor.from_pretrained("MCG-NJU/videomae-base")
    model = VideoMAEEncoderWithTemporalModel(
        pretrained_model_name="MCG-NJU/videomae-base",
        feature_dim=768,
        hidden_dim=512,
        temporal_model="bilstm",
        use_sliding_windows=True,
    ).to(device)

    # Process videos
    video_paths = video_files1[:5]  # Your video paths
    features = test_videomae_on_videos(
        model=model,
        image_processor=image_processor,
        video_paths=video_paths,
        batch_size=2,
        device=device,
    )

    print(f"Final features shape: {features.shape}")


## Feature extraction prototype

In [ ]:
import time


### `test_videomae_on_videos`

In [ ]:
def test_videomae_on_videos(
    model: nn.Module,
    image_processor,
    video_paths: List[str],
    batch_size: int = 2,
    device: str = "cuda",
    save_features: bool = False,
) -> Tuple[torch.Tensor, dict]:
    """
    Test the VideoMAE encoder with temporal modeling on real video data.
    """
    print("\n=== Starting VideoMAE Video Analysis ===")

    model = model.to(device)
    model.eval()

    metrics = {
        "processing_times": [],
        "memory_usage": [],
        "batch_sizes": [],
        "successful_videos": 0,
        "failed_videos": 0,
    }

    all_features = []

    for i in range(0, len(video_paths), batch_size):
        batch_paths = video_paths[i : i + batch_size]
        print(
            f"\nProcessing batch {i//batch_size + 1}/{(len(video_paths) + batch_size - 1)//batch_size}"
        )

        try:
            start_time = time.time()

            batch_inputs = []
            for video_path in batch_paths:
                # Process individual videos
                inputs = process_video_batch(video_path, image_processor)
                batch_inputs.append(inputs["pixel_values"])

            # Stack batch inputs
            video_batch = torch.cat(batch_inputs, dim=0).to(device)

            print("\n=== Batch Input Verification ===")
            print(f"Batch shape: {video_batch.shape}")
            print(f"Value range: [{video_batch.min():.3f}, {video_batch.max():.3f}]")
            print(f"Successfully processed {len(batch_paths)} videos")

            # Extract features
            with torch.no_grad():
                batch_features = model(pixel_values=video_batch)

            # Analyze features
            print("\n=== Feature Analysis ===")
            print(f"Feature shape: {batch_features.shape}")
            print(
                f"Feature range: [{batch_features.min():.3f}, {batch_features.max():.3f}]"
            )
            print(f"Feature mean: {batch_features.mean():.3f}")
            print(f"Feature std: {batch_features.std():.3f}")

            # Check for numerical stability
            if torch.isnan(batch_features).any():
                print("WARNING: NaN values detected in features!")
            if torch.isinf(batch_features).any():
                print("WARNING: Infinite values detected in features!")

            # Memory tracking
            if torch.cuda.is_available():
                memory_usage = torch.cuda.memory_allocated() / 1024**2
                print(f"\nCUDA Memory Usage: {memory_usage:.2f} MB")
                metrics["memory_usage"].append(memory_usage)

            # Save features if requested
            if save_features and output_dir:
                output_dir = Path(output_dir)
                output_dir.mkdir(parents=True, exist_ok=True)
                for j, path in enumerate(batch_paths):
                    feature_path = output_dir / f"features_batch{i}_video{j}.pt"
                    torch.save(batch_features[j], feature_path)

            # Update metrics
            process_time = time.time() - start_time
            metrics["processing_times"].append(process_time)
            metrics["batch_sizes"].append(len(batch_paths))
            metrics["successful_videos"] += len(batch_paths)

            all_features.append(batch_features.cpu())

            print(f"Batch processing time: {process_time:.2f} seconds")

        except Exception as e:
            print(f"Error processing batch: {str(e)}")
            metrics["failed_videos"] += len(batch_paths)
            continue

    # Combine all features
    if all_features:
        all_features = torch.cat(all_features, dim=0)
    else:
        raise ValueError("No features were successfully extracted")

    # Calculate final metrics
    metrics["total_time"] = sum(metrics["processing_times"])
    metrics["average_time_per_batch"] = np.mean(metrics["processing_times"])
    metrics["average_time_per_video"] = (
        metrics["total_time"] / metrics["successful_videos"]
    )
    if metrics["memory_usage"]:
        metrics["peak_memory_usage"] = max(metrics["memory_usage"])

    print("\n=== Final Processing Statistics ===")
    print(f"Total videos processed: {metrics['successful_videos']}")
    print(f"Failed videos: {metrics['failed_videos']}")
    print(f"Total processing time: {metrics['total_time']:.2f} seconds")
    print(f"Average time per video: {metrics['average_time_per_video']:.2f} seconds")
    if "peak_memory_usage" in metrics:
        print(f"Peak memory usage: {metrics['peak_memory_usage']:.2f} MB")

    return all_features, metrics


# Example usage


In [ ]:
if __name__ == "__main__":
    # Initialize components
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    image_processor = AutoImageProcessor.from_pretrained("MCG-NJU/videomae-base")
    model = VideoMAEEncoderWithTemporalModel(
        pretrained_model_name="MCG-NJU/videomae-base",
        feature_dim=768,
        hidden_dim=512,
        temporal_model="bilstm",
        use_sliding_windows=True,
    ).to(device)

    # Process videos
    video_paths = video_files1[:5]  # Your video paths
    features, metrics = test_videomae_on_videos(
        model=model,
        image_processor=image_processor,
        video_paths=video_paths,
        batch_size=2,
        device=device,
        save_features=True,
    )

    # Analyze temporal patterns
    analyze_temporal_patterns(features)


## Decoder and model wrapper

In [ ]:
import torch
import torch.nn as nn
from transformers import BertTokenizer, BertModel
from torch.nn import TransformerDecoder, TransformerDecoderLayer

from sign_language_lab.videomae_experiment import SignLanguageDecoder

# Complete model combining encoder and decoder
from sign_language_lab.videomae_experiment import SignLanguageTranslationModel

# Loss function
from sign_language_lab.losses import SignLanguageTranslationLoss


## Auxiliary losses and temporal attention

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from sign_language_lab.losses import LabelSmoothingLoss
from sign_language_lab.losses import ConsistencyLoss
from sign_language_lab.attention import TemporalAttention


## Development annotations

In [ ]:
# RETRIVING VIDEO_FILES, AND SENTENCES
import pandas as pd

video_files1 = glob.glob(
    os.path.join(val_dir, "*.mp4")
)  # Adjust extension as necessary

if video_files1:
    print(f"Found {len(video_files1)} video files.")
else:
    print("No video files found in the directory.")
    video_files1 = []  # Exit early if no video files found

# Step 2: Extract codes from video filenames
video_file_codes1 = [
    os.path.splitext(os.path.basename(video))[0].split("-")[-1].strip()
    for video in video_files1
]

# Step 3: Load Excel file and extract codes from the 'train' sheet
try:
    df_combined = pd.read_excel(
        excel_file, sheet_name="dev"
    )  # Adjust sheet name as necessary
    df_combined["code"] = df_combined["name"].apply(lambda x: x.split("-")[-1].strip())

    # Step 4: Match video file codes with codes from the Excel file
    matching_data = df_combined[df_combined["code"].isin(video_file_codes1)]

    if not matching_data.empty:
        matched_translations1 = []
        matched_video_files1 = []

        # Iterate over video file codes to match properly
        for video_code in video_file_codes1:
            matching_row = matching_data[matching_data["code"] == video_code]
            if not matching_row.empty:
                matched_translations1.append(matching_row["gloss"].iloc[0])
                matched_video_files1.append(video_code)

    else:
        # Handle cases with no matches
        unmatched_video_codes = [
            code
            for code in video_file_codes2
            if code not in df_combined["code"].tolist()
        ]

except FileNotFoundError:
    print(f"Error: Excel file '{excel_file}' not found.")
except KeyError as e:
    print(f"Error: Missing expected column in Excel file. Details: {e}")
# print((matched_translations1[:100]))
# print(len(video_files1[:5]))


## Historical training and validation loop

In [ ]:
from tqdm import tqdm
import traceback
import torch.nn.functional as F
from jiwer import wer
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from tqdm import tqdm
from torch.optim.lr_scheduler import OneCycleLR
from sklearn.model_selection import train_test_split


### `SignLanguageDataset`

In [ ]:
class SignLanguageDataset(Dataset):
    def __init__(self, video_paths, sentences):
        assert len(video_paths) == len(
            sentences
        ), "Number of videos and sentences must match"
        self.video_paths = video_paths
        self.sentences = sentences

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        return {
            "video_path": [
                self.video_paths[idx]
            ],  # Wrap in list for process_video_batch
            "text": self.sentences[idx],
        }


### `collate_fn`

In [ ]:
def collate_fn(batch):
    # Combine all video paths and sentences from the batch
    video_paths = []
    sentences = []

    for item in batch:
        video_paths.extend(item["video_path"])
        sentences.append(item["text"])

    return {"video_path": video_paths, "text": sentences}


### `AverageMeter`

In [ ]:
class AverageMeter:
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count


### `train_epoch`

In [ ]:
def train_epoch(
    model, dataloader, criterion, optimizer, scheduler, tokenizer, device, epoch
):
    model.train()
    losses = AverageMeter()
    gradient_accumulation_steps = 2
    optimizer.zero_grad()
    pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"Epoch {epoch}")
    for batch_idx, batch in pbar:
        try:
            # Process video batch using your function
            batch_videos = process_video_batch(
                batch["video_path"], clip_len=32, frame_sample_rate=1
            )
            batch_videos = batch_videos.to(device)
            texts = batch["text"]

            # Tokenize target text
            tokenized = tokenizer(
                texts, padding=True, truncation=True, return_tensors="pt"
            ).input_ids.to(device)

            # Forward pass
            logits = model(batch_videos, tokenized)

            # Calculate loss (ignore padding tokens)
            loss = criterion(logits[:, :-1].contiguous(), tokenized[:, 1:].contiguous())
            loss = loss / gradient_accumulation_steps
            loss.backward()

            if (batch_idx + 1) % gradient_accumulation_steps == 0 or (
                batch_idx + 1
            ) == len(dataloader):

                optimizer.step()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

                optimizer.zero_grad()

            # Update metrics
            scheduler.step()
            losses.update(
                loss.item() * gradient_accumulation_steps
            )  # Scale back for logging
            pbar.set_postfix({"loss": losses.avg})

        except Exception as e:
            print(f"Error in batch {batch_idx}: {str(e)}")
            continue

    return losses.avg


### `validate`

In [ ]:
@torch.no_grad()
def validate(model, dataloader, tokenizer, device, beam_size=5):
    model.eval()
    all_predictions = []
    all_targets = []
    char_wer_scores = []
    for batch in tqdm(dataloader, desc="Validating"):
        try:
            # Process video batch
            videos = process_video_file(
                batch["video_path"], clip_len=32, frame_sample_rate=1
            )
            videos = videos.to(device)
            texts = batch["text"]

            # Generate translations
            encoder_output = model.encoder(videos)
            predicted_tokens = model.decoder(encoder_output, beam_size=beam_size)

            # Convert tokens to text
            predictions = tokenizer.batch_decode(
                predicted_tokens, skip_special_tokens=True
            )

            all_predictions.extend(predictions)
            all_targets.extend(texts)
            for pred, target in zip(predictions, texts):
                # Convert strings to character sequences
                pred_chars = list(pred.lower().replace(" ", ""))
                target_chars = list(target.lower().replace(" ", ""))

                # Calculate character-level WER
                char_wer = wer([" ".join(target_chars)], [" ".join(pred_chars)])
                char_wer_scores.append(char_wer)
        except Exception as e:
            print(f"Error during validation: {str(e)}")
            traceback.print_exc()
            continue
    avg_char_wer = np.mean(char_wer_scores) if char_wer_scores else None
    return all_predictions, all_targets, {"char_wer": avg_char_wer}


### `main`

In [ ]:
def main():
    # Setup
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Your video paths and sentences
    video_paths = video_files1[:25]
    sentences = matched_translations1[:25]
    val_video_paths = video_files1[:25]
    val_sentences = matched_translations1[:25]
    # Create dataset and dataloader
    dataset = SignLanguageDataset(video_paths, sentences)
    val_dataset = SignLanguageDataset(val_video_paths, val_sentences)
    train_loader = DataLoader(
        dataset,
        batch_size=16,
        shuffle=True,
        num_workers=0,  # Set to 0 for debugging
        collate_fn=collate_fn,
        pin_memory=True if torch.cuda.is_available() else False,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=16,
        shuffle=False,
        num_workers=0,
        collate_fn=collate_fn,
        pin_memory=True,
    )
    # Initialize models
    encoder = VideoMAEEncoderWithTemporalModel(
        pretrained_model_name="MCG-NJU/videomae-base",
        feature_dim=768,
        hidden_dim=512,
        temporal_model="bilstm",
        use_sliding_windows=True,
    ).to(device)

    decoder = SignLanguageDecoder(
        encoder_dim=1024,
        bert_model_name="bert-base-german-cased",
    )

    model = SignLanguageTranslationModel(encoder, decoder).to(device)

    # Initialize optimizer and loss
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=1e-5,
        weight_decay=0.01,
        betas=(0.9, 0.999),
    )

    criterion = SignLanguageTranslationLoss(decoder.pad_token_id)
    """start_epoch = 22  # Starting from where we left off
    checkpoint = torch.load('best_model.pth')  # or 'best_model_wer.pth'
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    best_loss = checkpoint.get('loss', float('inf'))"""

    try:
        # Training loop
        num_epochs = 100
        best_loss = float("inf")
        total_steps = num_epochs * len(train_loader)
        scheduler = OneCycleLR(
            optimizer,
            max_lr=2e-4,
            total_steps=total_steps,
            pct_start=0.1,  # Warmup for first 10% of training
            anneal_strategy="linear",
        )
        for epoch in range(num_epochs):
            # Train
            train_loss = train_epoch(
                model=model,
                dataloader=train_loader,
                criterion=criterion,
                optimizer=optimizer,
                scheduler=scheduler,
                tokenizer=decoder.tokenizer,
                device=device,
                epoch=epoch,
            )

            # Print progress
            print(f"\nEpoch {epoch+1}/{num_epochs}")
            print(f"Training Loss: {train_loss:.4f}")

            # Validate and print example translations
            if (epoch + 1) % 1 == 0:  # Validate every 5 epochs
                predictions, targets, metrics = validate(
                    model=model,
                    dataloader=val_loader,
                    tokenizer=decoder.tokenizer,
                    device=device,
                    beam_size=5,
                )
                if metrics and metrics.get("char_wer") is not None:
                    print(f"Character-level WER: {metrics['char_wer']:.4f}")
                else:
                    print("Could not calculate WER")
                print("\nExample translations:")
                for pred, target in zip(predictions[:2], targets[:2]):
                    print(f"Predicted: {pred}")
                    print(f"Target: {target}")
                    print()

            # Save best model
            if train_loss < best_loss:
                best_loss = train_loss
                torch.save(
                    {
                        "epoch": epoch,
                        "model_state_dict": model.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "loss": train_loss,
                    },
                    "best_model_mae.pth",
                )
                print(f"Saved new best model with loss: {best_loss:.4f}")

    except Exception as e:
        print(f"An error occurred: {str(e)}")
        raise e
    finally:
        print("Training completed or interrupted.")


In [ ]:
if __name__ == "__main__":
    main()
